# T16 — CD-HIT-grouped split vs. random split, measured at short and full length

**Why this notebook exists.** A reviewer pointed out two real gaps in the leakage analysis (T1 /
Table 7): (a) T1's random-split arm doesn't use the MinHash cluster tags at all, so a near-zero
random-vs-cluster delta there does not prove clustering "removed" leakage -- it only shows that,
on THIS dataset, near-duplicates barely matter; and (b) T1 only measured the gap at full length,
where every model is already near-ceiling (>0.97 macro-F1) and small differences are noisy. The
place leakage is most likely to matter is at short fragment lengths, and the paper's own
Section 4.9 shows the MinHash clustering collapses at any threshold looser than J>=0.90 -- but we
already have CD-HIT clusters at 95% and 97% identity from T2, which do NOT collapse (CD-HIT
doesn't chain the way connected-component MinHash clustering does).

**What this notebook does.** It re-derives CD-HIT clusters at 95% and 97% identity (same install
+ command as T2), then uses those CD-HIT cluster ids -- not the MinHash cluster ids -- as the
`groups` argument to `StratifiedGroupKFold`, and compares that split against an ordinary random
`StratifiedKFold` split, for every classifier including the three CNNs, at 300 bp, 500 bp, and
full length.

**This notebook is written to run unattended end to end.** The CNN section (Section 8) is not a
placeholder: it reuses the exact same 1D VGG-16 / ResNet-18 / Inception-v1 Keras architectures and
training loop as T1, so you can Run All and walk away. The one thing to check before you do: the
full grid (2 identity levels x 2 split arms x 3 lengths x 3 seeds x 3 CNNs = 108 CNN fits) is the
most expensive run in this whole project, realistically several hours of GPU time. Read the
runtime note in Section 2 and set `FAST_MODE` or `RUN_DEEP_MODELS` there BEFORE you commit and
leave, since those are the two knobs that trade completeness for time and you cannot easily change
them once a long unattended run is already underway.

**GPU needed.** Yes, for Section 8 (the three CNNs). Sections 1-7 (CD-HIT clustering plus SVM and
random forest) run fine on CPU and do not need the accelerator on at all.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports and install CD-HIT (same as T2 -- installed here, nothing to download yourself)

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, subprocess, shutil, random
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)


### 2. Config -- READ THIS before you Run All and walk away

`RUN_DEEP_MODELS = True` plus `FAST_MODE = False` runs the full grid: 108 CNN fits, realistically 4-9 hours of GPU time on top of the CPU-only classical section. If that does not fit your remaining GPU budget, set `FAST_MODE = True` now (2 seeds, one identity level, halves the grid) or `RUN_DEEP_MODELS = False` (classical only, no GPU needed at all, finishes in under an hour) -- both are safe to leave unattended, they just produce a smaller result.

In [ ]:

FAST_MODE       = False          # True: 2 seeds, only 97% identity (roughly halves total runtime)
SEEDS           = [0, 1] if FAST_MODE else [0, 1, 2]
IDENTITIES      = [0.97] if FAST_MODE else [0.95, 0.97]
RUN_DEEP_MODELS = True            # False: classical-only (SVM/RF), no GPU needed, finishes in <1h
LENGTHS         = [300, 500, None]   # None = full length
MAX_TFIDF_FEATS = 512
DEEP_MODELS     = ["resnet18", "inception"] if FAST_MODE else ["resnet18", "vgg16", "inception"]


### 3. Load data and write FASTA for CD-HIT

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)

le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
n_cls = len(le.classes_)
X_all = df["sequence"].values
n = len(df)
print(f"{n} sequences, {n_cls} classes")

with open(f"{OUT}/pol_t16.fasta", "w") as f:
    for i, s in enumerate(X_all):
        f.write(f">{i}\n{s}\n")


### 4. Run CD-HIT at each identity level and parse cluster assignments

In [ ]:

def run_cdhit(identity, word_size):
    out = f"{OUT}/pol_t16_cdhit_{int(identity*100)}"
    cmd = f"cd-hit-est -i {OUT}/pol_t16.fasta -o {out} -c {identity} -n {word_size} -M 16000 -T 4 -d 0"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return out, r.returncode

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all(), "some sequences never got a cluster id -- CD-HIT output does not match the input FASTA"
    return labels

WORD_SIZE = {0.95: 10, 0.97: 10}   # same word size T2 used, kept identical for consistency
cdhit_groups = {}
for identity in IDENTITIES:
    out, rc = run_cdhit(identity, WORD_SIZE[identity])
    assert rc == 0 and os.path.exists(out + ".clstr"), f"CD-HIT failed at identity={identity} (returncode={rc})"
    labs = parse_cdhit_clusters(out + ".clstr", n)
    cdhit_groups[identity] = labs
    print(f"CD-HIT {identity*100:.0f}% identity: {len(set(labs))} clusters, "
          f"largest = {pd.Series(labs).value_counts().iloc[0]} sequences")


### 5. Fragmentation helper (same convention as the other notebooks)

In [ ]:

def fragment(seq, L, rng):
    if L is None or len(seq) <= L:
        return seq
    start = rng.integers(0, len(seq) - L)
    return seq[start:start+L]


### 6. Split helpers -- one random arm, one CD-HIT-grouped arm per identity level, each carving its own validation fold for the CNNs

In [ ]:

def split_random(X, y, seed):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    skf2 = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
    tr2, va2 = next(skf2.split(X[tr_idx], y[tr_idx]))
    return tr_idx[tr2], tr_idx[va2], te_idx

def split_grouped(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te


### 7. Classical models (SVM, RF) at every length, both split arms -- CPU only

In [ ]:

def fit_tfidf(seq_tr, seq_va, seq_te):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=MAX_TFIDF_FEATS, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xva = vec.transform(seq_va); Xte = vec.transform(seq_te)
    return Xtr, Xva, Xte

def fit_score_classical(Xtr, ytr, Xte, yte, model_name, seed):
    if model_name == "svm":
        clf = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)
    elif model_name == "rf":
        clf = RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=seed, n_jobs=-1)
    else:
        raise ValueError(model_name)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    return f1_score(yte, pred, average="macro"), balanced_accuracy_score(yte, pred)

CLASSICAL = ["svm", "rf"]
rows = []
splits_cache = {}   # (tag, seed, arm, identity) -> (tr, va, te) index arrays, reused by the CNN section below

for L in LENGTHS:
    tag = L if L else "full"
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        seqs = np.array([fragment(s, L, rng) for s in X_all])

        tr, va, te = split_random(seqs, y_all, seed)
        splits_cache[(tag, seed, "random", None)] = (tr, va, te)
        Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
        for m in CLASSICAL:
            f1, bal = fit_score_classical(Xtr, y_all[tr], Xte, y_all[te], m, seed)
            rows.append(dict(length=tag, seed=seed, arm="random", identity=None, model=m, f1=f1, bal_acc=bal))

        for identity, groups in cdhit_groups.items():
            tr, va, te = split_grouped(seqs, y_all, groups, seed)
            splits_cache[(tag, seed, "cdhit_grouped", identity)] = (tr, va, te)
            Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
            for m in CLASSICAL:
                f1, bal = fit_score_classical(Xtr, y_all[tr], Xte, y_all[te], m, seed)
                rows.append(dict(length=tag, seed=seed, arm="cdhit_grouped", identity=identity,
                                  model=m, f1=f1, bal_acc=bal))
        pd.DataFrame(rows).to_csv(f"{OUT}/t16_cdhit_split_classical_raw.csv", index=False)  # incremental save
        print(f"[classical] length={tag} seed={seed} done")

t16_classical = pd.DataFrame(rows)
t16_classical.groupby(["length","arm","identity","model"])[["f1","bal_acc"]].agg(["mean","std"]).round(4)


### 8. CNNs at every length, both split arms -- needs GPU. Same architectures and training loop as T1, so results are directly comparable to Table 7.

In [ ]:

if RUN_DEEP_MODELS:
    import tensorflow as tf
    from tensorflow.keras import layers, Model, Input

    def conv_bn_relu(x, filters, kernel=5, stride=1):
        x = layers.Conv1D(filters, kernel, strides=stride, padding="same")(x)
        x = layers.BatchNormalization()(x)
        return layers.Activation("relu")(x)

    def build_vgg16_1d(input_len, n_classes):
        inp = Input(shape=(input_len, 1)); x = inp
        for filters, n_convs in [(32,2),(64,2),(128,3),(256,3),(256,3)]:
            for _ in range(n_convs):
                x = conv_bn_relu(x, filters, 3)
            x = layers.MaxPooling1D(2, padding="same")(x)
        x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)
        return Model(inp, out, name="vgg16_1d")

    class StochasticDepth(layers.Layer):
        def __init__(self, p=0.85, **kw):
            super().__init__(**kw); self.p = p
        def call(self, x, training=None):
            if not training:
                return x
            r = tf.random.uniform([tf.shape(x)[0], 1, 1])
            return x * tf.floor(self.p + r) / self.p
        def get_config(self):
            return {**super().get_config(), "p": self.p}

    def res_block(x, filters, stride=1):
        shortcut = x
        y = conv_bn_relu(x, filters, 3, stride)
        y = layers.Conv1D(filters, 3, padding="same")(y); y = layers.BatchNormalization()(y)
        if shortcut.shape[-1] != filters or stride != 1:
            shortcut = layers.Conv1D(filters, 1, strides=stride, padding="same")(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        y = StochasticDepth(0.85)(y)
        return layers.Activation("relu")(layers.Add()([shortcut, y]))

    def build_resnet18_1d(input_len, n_classes):
        inp = Input(shape=(input_len, 1))
        x = conv_bn_relu(inp, 32, 7, 2); x = layers.MaxPooling1D(3, strides=2, padding="same")(x)
        for filters, stride in [(32,1),(32,1),(64,2),(64,1),(128,2),(128,1),(256,2),(256,1)]:
            x = res_block(x, filters, stride)
        x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)
        return Model(inp, out, name="resnet18_1d")

    def inception_module(x, f1x1, f3x3, f5x5, fpool):
        b1 = conv_bn_relu(x, f1x1, 1); b2 = conv_bn_relu(x, f3x3, 3); b3 = conv_bn_relu(x, f5x5, 5)
        b4 = layers.MaxPooling1D(3, strides=1, padding="same")(x); b4 = conv_bn_relu(b4, fpool, 1)
        return layers.Concatenate()([b1, b2, b3, b4])

    def build_inception_1d(input_len, n_classes):
        inp = Input(shape=(input_len, 1))
        x = conv_bn_relu(inp, 32, 7, 2); x = layers.MaxPooling1D(3, strides=2, padding="same")(x)
        x = inception_module(x, 16, 24, 8, 8); x = inception_module(x, 32, 40, 16, 16)
        x = layers.MaxPooling1D(2, padding="same")(x); x = inception_module(x, 40, 48, 20, 20)
        x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)
        return Model(inp, out, name="inception_v1_1d")

    BUILDERS = {"vgg16": build_vgg16_1d, "resnet18": build_resnet18_1d, "inception": build_inception_1d}

    def run_deep(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, epochs=60, batch=48):
        tf.random.set_seed(seed)
        Xtr_d = np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr, dtype=np.float32)[..., None]
        Xva_d = np.asarray(Xva.todense() if hasattr(Xva, "todense") else Xva, dtype=np.float32)[..., None]
        Xte_d = np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte, dtype=np.float32)[..., None]
        mu, sd = Xtr_d.mean(), Xtr_d.std() + 1e-8
        Xtr_d = (Xtr_d - mu) / sd; Xva_d = (Xva_d - mu) / sd; Xte_d = (Xte_d - mu) / sd
        model = BUILDERS[model_name](Xtr_d.shape[1], n_cls)
        cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                       loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
        es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        model.fit(Xtr_d, ytr, validation_data=(Xva_d, yva), epochs=epochs, batch_size=batch,
                  class_weight=cw, callbacks=[es], verbose=0)
        pred = np.argmax(model.predict(Xte_d, verbose=0), axis=1)
        f1 = f1_score(yte, pred, average="macro"); bal = balanced_accuracy_score(yte, pred)
        tf.keras.backend.clear_session()
        return f1, bal

    deep_rows = []
    for (tag, seed, arm, identity), (tr, va, te) in splits_cache.items():
        L = None if tag == "full" else tag
        rng = np.random.default_rng(seed)
        seqs = np.array([fragment(s, L, rng) for s in X_all])
        Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
        for m in DEEP_MODELS:
            f1, bal = run_deep(m, Xtr, y_all[tr], Xva, y_all[va], Xte, y_all[te], seed)
            deep_rows.append(dict(length=tag, seed=seed, arm=arm, identity=identity, model=m, f1=f1, bal_acc=bal))
        pd.DataFrame(deep_rows).to_csv(f"{OUT}/t16_cdhit_split_deep_raw.csv", index=False)  # incremental save
        print(f"[deep] length={tag} seed={seed} arm={arm} identity={identity} done")

    t16_deep = pd.DataFrame(deep_rows)
    t16_all = pd.concat([t16_classical, t16_deep], ignore_index=True)
else:
    print("RUN_DEEP_MODELS is False -- skipped. t16_all is classical-only.")
    t16_all = t16_classical

t16_all.to_csv(f"{OUT}/t16_all_raw.csv", index=False)


### 9. Summary table matching the format of Table 7 (delta = random minus cdhit_grouped)

In [ ]:

summary = (t16_all.groupby(["length","arm","identity","model"])["f1"].mean()
           .reset_index().pivot_table(index=["length","model"], columns=["arm","identity"], values="f1"))
summary.to_csv(f"{OUT}/t16_summary_table.csv")
print(summary.round(4))
print("""
Read this table by row (one length x one model). A delta near the T1/Table 7 full-length delta
(under 0.01) at 300 and 500 bp too means leakage genuinely doesn't matter much at this task even
under fragmentation. A delta that GROWS at short lengths means the honest leakage story is
different from what Table 7 alone suggested, and Table 7 / Section 4.1 / the Introduction need to
be rewritten around this new, more complete result.
""")


In [ ]:
!cd /kaggle/working && zip -r t16_cdhit_split_outputs.zip t16_all_raw.csv t16_summary_table.csv t16_cdhit_split_classical_raw.csv